# Using Matrix Factorization for Counterfactual Species Pool Reconstruction

### 1. The Challenge: Quantifying the Unseen
Estimating **Dark Diversity**—the set of species absent from a site despite suitable conditions—requires reconstructing the **Species Pool** (the "Potential" distribution). Traditional methods often suffer from:

* **Environmental Confounding:** Failing to distinguish between shared environmental preferences and actual biotic interactions.
* **Subjectivity in Benchmarking:** Forcing researchers to manually define what a "natural" or "pristine" value looks like for complex variables such as canopy height or nutrient levels.
* **Data Sparsity:** Relying on pairwise co-occurrences that become unreliable in hyper-diverse, sparse communities.

### 2. The Solution: Residual-based Latent Counterfactuals
This implementation utilizes a **Joint Species Distribution Model (JSDM)** framework. Instead of manually defining every possible driver, we use **Matrix Factorization (MF)** to partition species distributions into two distinct mathematical components:

1.  **Abiotic Fixed Effects ($\mathbf{x}_i^\top \boldsymbol{\beta}_j$):** This represents the **Potential Distribution**. It captures the species' response to measured, fundamental environmental variables like Temperature, pH, and Elevation.
2.  **Latent Random Effects ($\mathbf{w}_i^\top \mathbf{z}_j$):** This acts as a "statistical vacuum cleaner," capturing all unmeasured deviations from the abiotic niche, such as land-use degradation, biotic competition, or dispersal limitation.

**The Integrated Model Equation:**
$$\text{logit}(p_{ij}) = \alpha_j + \mathbf{x}_i^\top \boldsymbol{\beta}_j + \mathbf{w}_i^\top \mathbf{z}_j$$

### 3. Reconstructing the Species Pool via Counterfactuals
The core of this method is the **Counterfactual Prediction**. To reconstruct the original "Potential" distribution (the Species Pool) without making subjective judgment calls on degradation, we:

* **Train with Latents:** Include the latent factors ($W, Z$) during training so they "soak up" the noise and stressors, preventing them from biasing the abiotic coefficients ($\beta$).
* **Predict without Latents:** Generate predictions using only the abiotic layer ($\alpha + X\beta$), effectively setting the unmeasured stressors to zero (the prior mean). 

This allows us to mathematically "undo" the impact of the unmeasured drivers of dark diversity to reveal the underlying environmental suitability.

### 4. Scalable Inference via SVI
To handle thousands of sites and species, we utilize **Stochastic Variational Inference (SVI)** in Pyro. SVI treats inference as an optimization problem, minimizing the difference between our approximation and the true posterior (ELBO optimization), which allows for efficient computation on high-dimensional ecological matrices even with limited computational resources.

In [14]:
import pandas as pd
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

In [15]:
import os
from pathlib import Path

# Create output directory if it doesn't exist
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
print(f"Output directory: {output_dir.resolve()}")

Output directory: C:\Users\dyshe\OneDrive - Aarhus universitet\Documents\PhD\Chapter 1 - Biodiversity and restoration potential\Simulation_study\Matrix factorisation\output


In [16]:
# 1. Load and Prepare Data
survey_full_df = pd.read_csv('C:/Users/dyshe/OneDrive - Aarhus universitet/Documents/PhD/Chapter 1 - Biodiversity and restoration potential/Simulation_study/survey.csv', index_col=0)
# Extract spatial coordinates before dropping them
spatial_coords = survey_full_df[['x', 'y']].values
survey_df = survey_full_df.drop(columns=['ID', 'x', 'y'])

env_df = pd.read_csv('C:/Users/dyshe/OneDrive - Aarhus universitet/Documents/PhD/Chapter 1 - Biodiversity and restoration potential/Simulation_study/env.csv', index_col=0)       # sites x env
# Drop ID and landuse columns - keep only abiotic variables
env_df = env_df.drop(columns=['ID', 'landuse'])

# Rescale the abiotic env data to have mean 0 and std 1
continuous_cols = env_df.columns.tolist()
env_df[continuous_cols] = (env_df[continuous_cols] - env_df[continuous_cols].mean()) / env_df[continuous_cols].std()

# Ensure all data is float type
env_df = env_df.astype(float)

# Convert to Tensors
# Y is the observation matrix (binary 0/1)
Y = torch.tensor(survey_df.values, dtype=torch.float32)
# X is the environmental matrix (Standardized abiotic variables only)
X = torch.tensor(env_df.values, dtype=torch.float32)
X = (X - X.mean(dim=0)) / (X.std(dim=0) + 1e-8)  # Add small epsilon to avoid division by zero

num_sites, num_species = Y.shape
num_env = X.shape[1]
num_factors = 5  # Number of latent factors to capture unmeasured drivers

print(f"Environmental variables: {env_df.columns.tolist()}")
print(f"Number of environmental covariates: {num_env}")
print(f"Number of latent factors: {num_factors}")
print(f"Spatial coordinates shape: {spatial_coords.shape}")


Environmental variables: ['temperature', 'pH', 'elevation']
Number of environmental covariates: 3
Number of latent factors: 5
Spatial coordinates shape: (225, 2)


In [17]:
# 2. Define the Model
def model(X, Y, num_factors):
    n_sites, n_species = Y.shape
    n_env = X.shape[1]

    # Priors for species-specific intercepts (baseline prevalence)
    alpha = pyro.sample("alpha", dist.Normal(0, 1).expand([n_species]).to_event(1))

    # Priors for Environmental coefficients (Beta)
    # Each species has its own response to each environmental variable
    beta = pyro.sample("beta", dist.Normal(0, 1).expand([n_env, n_species]).to_event(2))

    # Matrix Factorization Components
    # W: Site latent scores (The "environment" we didn't measure)
    W = pyro.sample("W", dist.Normal(0, 1).expand([n_sites, num_factors]).to_event(2))
    # Z: Species latent loadings (How species respond to unmeasured factors)
    Z = pyro.sample("Z", dist.Normal(0, 1).expand([n_species, num_factors]).to_event(2))

    # Linear Predictor: Intercept + Env Effects + Latent Factor Interaction
    # eta shape: (n_sites, n_species)
    eta = alpha + torch.matmul(X, beta) + torch.matmul(W, Z.t())

    # Likelihood: Bernoulli with Logit link
    with pyro.plate("sites", n_sites):
        pyro.sample("obs", dist.Bernoulli(logits=eta).to_event(1), obs=Y)

In [18]:
# 3. Setup Inference
pyro.clear_param_store()
guide = AutoDiagonalNormal(model)
optimizer = Adam({"lr": 0.01})
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

In [19]:
# 4. Training Loop
num_iterations = 2500
for i in range(num_iterations):
    loss = svi.step(X, Y, num_factors)
    if i % 500 == 0:
        print(f"Iteration {i} - Loss: {loss:.2f}")

Iteration 0 - Loss: 20587.98
Iteration 500 - Loss: 6879.62
Iteration 1000 - Loss: 6638.01
Iteration 1500 - Loss: 6563.67
Iteration 2000 - Loss: 6539.01


In [20]:
# 5. Predict Species Distributions
# Get the "MAP" (Maximum A Posteriori) estimates from the guide
medians = guide.median()
alpha_hat = medians['alpha']
beta_hat = medians['beta']
W_hat = medians['W']
Z_hat = medians['Z']

# Calculate Predicted Probabilities
with torch.no_grad():
    # Full predictions: includes both environmental effects and latent factors
    logits_full = alpha_hat + torch.matmul(X, beta_hat) + torch.matmul(W_hat, Z_hat.t())
    probs_full = torch.sigmoid(logits_full)
    
    # Environmental-only predictions: excludes latent factors
    # This represents species' responses to measured environmental variables only
    logits_env_only = alpha_hat + torch.matmul(X, beta_hat)
    probs_env_only = torch.sigmoid(logits_env_only)

# Full predictions (observed diversity with all drivers)
predicted_probs_full_df = pd.DataFrame(probs_full.numpy(), index=survey_df.index, columns=survey_df.columns)
predicted_probs_full_df.to_csv(output_dir / 'mat_fact_predicted_probabilities_full.csv')

# Environmental-only predictions (potential diversity without unmeasured drivers)
predicted_probs_env_only_df = pd.DataFrame(probs_env_only.numpy(), index=survey_df.index, columns=survey_df.columns)
predicted_probs_env_only_df.to_csv(output_dir / 'mat_fact_predicted_probabilities_env_only.csv')

# Dark diversity proxy: difference between full and environmental-only
dark_diversity_proxy_df = predicted_probs_full_df - predicted_probs_env_only_df
dark_diversity_proxy_df.to_csv(output_dir / 'mat_fact_dark_diversity_proxy.csv')

print("Predictions generated:")
print(f"Full model (environment + latent): {predicted_probs_full_df.shape}")
print(f"  Mean probability: {predicted_probs_full_df.values.mean():.4f}")
print(f"\nEnvironment-only (no latent): {predicted_probs_env_only_df.shape}")
print(f"  Mean probability: {predicted_probs_env_only_df.values.mean():.4f}")
print(f"\nDark diversity proxy (full - env_only):")
print(f"  Mean difference: {dark_diversity_proxy_df.values.mean():.4f}")

Predictions generated:
Full model (environment + latent): (225, 100)
  Mean probability: 0.0937

Environment-only (no latent): (225, 100)
  Mean probability: 0.1814

Dark diversity proxy (full - env_only):
  Mean difference: -0.0877


## 5b. Spatial Autocorrelation Model: Matrix Factorization with Thin Plate Spline Random Effects

To account for spatial autocorrelation in species distributions, we extend the baseline matrix factorization model with a **spatially-structured random effect** using **Thin Plate Splines (TPS)**. This allows us to capture unmeasured spatial patterns that are not explained by environmental variables or the latent factors.

### Model Specification

The spatial model decomposes species occurrence probabilities into four additive components:

$$\text{logit}(p_{ij}) = \underbrace{\alpha_j}_{\text{Intercept}} + \underbrace{\mathbf{x}_i^\top \boldsymbol{\beta}_j}_{\text{Environmental Effects}} + \underbrace{\mathbf{w}_i^\top \mathbf{z}_j}_{\text{Latent Factors}} + \underbrace{f(s_i)}_{\text{Spatial Random Effect}}$$

Where:
- **$\alpha_j$**: Species-specific baseline prevalence (intercept)
- **$\mathbf{x}_i^\top \boldsymbol{\beta}_j$**: Environmental contribution—species $j$'s response to abiotic variables at site $i$
- **$\mathbf{w}_i^\top \mathbf{z}_j$**: Latent factor contribution—species-specific biotic/dispersal effects
- **$f(s_i)$**: **NEW** Spatial random effect—captures spatially-autocorrelated deviations

### Spatial Random Effect via Thin Plate Splines

The spatial component is modeled as a weighted sum of radial basis functions:

$$f(s_i) = \sum_{k=1}^{K} \phi_k(s_i) \, \gamma_{kj}$$

Where:
- **$\phi_k(s_i) = r_{ik}^2 \log(r_{ik})$**: Thin plate spline radial basis function (distance $r_{ik}$ between site $i$ and knot $k$)
- **$\gamma_{kj}$**: Weight for basis function $k$ and species $j$ (estimated coefficients)
- **$K$**: Number of basis functions (knots), selected via k-means clustering of spatial coordinates
- Additional **polynomial trend terms** (1, $x$, $y$) ensure the model captures linear and constant spatial gradients

### Interpretation of Components

1. **Environmental Layer** ($\alpha + X\beta$): Represents the "potential" distribution under ideal conditions
2. **Latent + Spatial Layers** ($W Z^T + f(s)$): Capture degradation, dispersal limitation, and other unmeasured spatially-structured factors
3. **Counterfactual Predictions**: Setting $W Z^T + f(s) = 0$ yields environment-only predictions for the species pool

### Advantages of This Approach

- **Explicit spatial modeling**: TPS efficiently captures smooth spatial gradients without assuming a particular spatial structure (e.g., Markov Random Field assumptions)
- **Flexible smoothness**: The number of knots controls the smoothness—fewer knots for smoother patterns, more knots for local variation
- **Species-specific spatial patterns**: Each species has its own spatial effect ($\gamma_{·j}$), allowing heterogeneous spatial responses

In [21]:
# ============================================================================
# SPATIAL AUTOCORRELATION MODEL - Thin Plate Spline Random Effects
# ============================================================================

# 6. Create Thin Plate Spline Basis Functions for Spatial Effects
def compute_tps_basis(coords, num_knots=None, regularization=0.01):
    """
    Compute thin plate spline (TPS) basis functions for spatial smoothing.
    
    Args:
        coords: (n_sites, 2) array of spatial coordinates
        num_knots: Number of basis functions (knots). If None, use sqrt(n_sites)
        regularization: Ridge regression parameter for stability
    
    Returns:
        phi: (n_sites, num_basis) basis matrix including intercept and linear terms
    """
    n_sites = coords.shape[0]
    
    if num_knots is None:
        num_knots = max(int(np.sqrt(n_sites)), 5)
    
    # Select knot locations using k-means clustering
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=num_knots, random_state=42, n_init=10)
    knots = kmeans.fit_predict(coords)
    knot_coords = kmeans.cluster_centers_
    
    # Compute pairwise distances between sites and knots
    distances = cdist(coords, knot_coords, metric='euclidean')
    
    # Thin plate spline radial basis function: r^2 * log(r)
    # Handle r=0 case to avoid log(0)
    phi_rbf = np.zeros((n_sites, num_knots))
    for i in range(num_knots):
        r = distances[:, i]
        # r^2 * log(r), with r=0 terms set to 0
        phi_rbf[:, i] = np.where(r > 0, r**2 * np.log(r), 0)
    
    # Add polynomial trend terms (1, x, y) for completeness
    poly_terms = np.column_stack([np.ones(n_sites), coords[:, 0], coords[:, 1]])
    
    # Combine basis functions
    phi = np.column_stack([poly_terms, phi_rbf])
    
    # Apply Tikhonov ridge regularization for numerical stability
    phi = phi + regularization * np.random.randn(*phi.shape) * 1e-6
    
    return torch.tensor(phi, dtype=torch.float32), torch.tensor(knot_coords, dtype=torch.float32)

# Compute TPS basis
phi_tps, knot_coords = compute_tps_basis(spatial_coords, num_knots=10)
num_tps_basis = phi_tps.shape[1]

print(f"TPS basis functions: {num_tps_basis}")
print(f"Knot coordinates shape: {knot_coords.shape}")


TPS basis functions: 13
Knot coordinates shape: torch.Size([10, 2])


In [22]:
# 7. Define Spatial Model with TPS Random Effects
def model_spatial(X, Y, phi_tps, num_factors):
    """
    Matrix factorization model with spatial random effects (thin plate spline).
    
    The spatial effect is modeled as:
    f(s_i) = sum_k phi_k(s_i) * gamma_k
    where phi_k are TPS basis functions and gamma_k are weights
    
    Full predictor: logit(p_ij) = alpha_j + x_i^T beta_j + w_i^T z_j + f(s_i)
    """
    n_sites, n_species = Y.shape
    n_env = X.shape[1]
    n_tps = phi_tps.shape[1]
    
    # Priors for species-specific intercepts
    alpha = pyro.sample("alpha", dist.Normal(0, 1).expand([n_species]).to_event(1))
    
    # Priors for Environmental coefficients
    beta = pyro.sample("beta", dist.Normal(0, 1).expand([n_env, n_species]).to_event(2))
    
    # Matrix Factorization Components (same as original)
    W = pyro.sample("W", dist.Normal(0, 1).expand([n_sites, num_factors]).to_event(2))
    Z = pyro.sample("Z", dist.Normal(0, 1).expand([n_species, num_factors]).to_event(2))
    
    # Spatial Random Effects: TPS coefficients
    # gamma has shape (n_tps, n_species) - each species has its own spatial pattern
    gamma = pyro.sample("gamma", dist.Normal(0, 0.5).expand([n_tps, n_species]).to_event(2))
    
    # Compute spatial effects: (n_sites, n_species)
    spatial_effects = torch.matmul(phi_tps, gamma)
    
    # Linear Predictor: Intercept + Env + Latent + Spatial
    eta = alpha + torch.matmul(X, beta) + torch.matmul(W, Z.t()) + spatial_effects
    
    # Likelihood: Bernoulli with Logit link
    with pyro.plate("sites", n_sites):
        pyro.sample("obs", dist.Bernoulli(logits=eta).to_event(1), obs=Y)


In [23]:
# 8. Setup Inference for Spatial Model
pyro.clear_param_store()
guide_spatial = AutoDiagonalNormal(model_spatial)
optimizer_spatial = Adam({"lr": 0.01})
svi_spatial = SVI(model_spatial, guide_spatial, optimizer_spatial, loss=Trace_ELBO())

print("Spatial model setup complete. Starting training...")


Spatial model setup complete. Starting training...


In [24]:
# 9. Training Loop for Spatial Model
num_iterations = 6000
for i in range(num_iterations):
    loss = svi_spatial.step(X, Y, phi_tps, num_factors)
    if i % 500 == 0:
        print(f"Iteration {i} - Loss: {loss:.2f}")


Iteration 0 - Loss: 86107324.89
Iteration 500 - Loss: 5932044.67
Iteration 1000 - Loss: 3538405.32
Iteration 1500 - Loss: 2330965.07
Iteration 2000 - Loss: 1404073.97
Iteration 2500 - Loss: 1191714.63
Iteration 3000 - Loss: 966466.39
Iteration 3500 - Loss: 779858.75
Iteration 4000 - Loss: 664846.53
Iteration 4500 - Loss: 590607.38
Iteration 5000 - Loss: 494214.23
Iteration 5500 - Loss: 456357.70


In [25]:
# 10. Predict Species Distributions (Spatial Model)
medians_spatial = guide_spatial.median()
alpha_hat_spatial = medians_spatial['alpha']
beta_hat_spatial = medians_spatial['beta']
W_hat_spatial = medians_spatial['W']
Z_hat_spatial = medians_spatial['Z']
gamma_hat_spatial = medians_spatial['gamma']

# Calculate spatial effects
spatial_effects_spatial = torch.matmul(phi_tps, gamma_hat_spatial)

# Calculate Predicted Probabilities (Spatial Model)
with torch.no_grad():
    # Full predictions: includes environmental, latent, and spatial effects
    logits_full_spatial = (alpha_hat_spatial + torch.matmul(X, beta_hat_spatial) + 
                           torch.matmul(W_hat_spatial, Z_hat_spatial.t()) + spatial_effects_spatial)
    probs_full_spatial = torch.sigmoid(logits_full_spatial)
    
    # Environmental-only predictions: excludes latent and spatial factors
    logits_env_only_spatial = alpha_hat_spatial + torch.matmul(X, beta_hat_spatial)
    probs_env_only_spatial = torch.sigmoid(logits_env_only_spatial)

# Full predictions (spatial model)
predicted_probs_full_spatial_df = pd.DataFrame(probs_full_spatial.numpy(), index=survey_df.index, columns=survey_df.columns)
predicted_probs_full_spatial_df.to_csv(output_dir / 'mat_fact_predicted_probabilities_full_spatial.csv')

# Environmental-only predictions (spatial model)
predicted_probs_env_only_spatial_df = pd.DataFrame(probs_env_only_spatial.numpy(), index=survey_df.index, columns=survey_df.columns)
predicted_probs_env_only_spatial_df.to_csv(output_dir / 'mat_fact_predicted_probabilities_env_only_spatial.csv')

# Dark diversity proxy (spatial model)
dark_diversity_proxy_spatial_df = predicted_probs_full_spatial_df - predicted_probs_env_only_spatial_df
dark_diversity_proxy_spatial_df.to_csv(output_dir / 'mat_fact_dark_diversity_proxy_spatial.csv')

print("Spatial model predictions generated:")
print(f"Full model (environment + latent + spatial): {predicted_probs_full_spatial_df.shape}")
print(f"  Mean probability: {predicted_probs_full_spatial_df.values.mean():.4f}")
print(f"\nEnvironment-only (spatial model): {predicted_probs_env_only_spatial_df.shape}")
print(f"  Mean probability: {predicted_probs_env_only_spatial_df.values.mean():.4f}")
print(f"\nDark diversity proxy (spatial model):")
print(f"  Mean difference: {dark_diversity_proxy_spatial_df.values.mean():.4f}")


Spatial model predictions generated:
Full model (environment + latent + spatial): (225, 100)
  Mean probability: 0.0901

Environment-only (spatial model): (225, 100)
  Mean probability: 0.4932

Dark diversity proxy (spatial model):
  Mean difference: -0.4030


In [26]:
# 11. Model Comparison: Non-Spatial vs Spatial
import sklearn.metrics as metrics

# Flatten predictions and observations for comparison
Y_flat = Y.numpy().flatten()
probs_full_flat = probs_full.numpy().flatten()
probs_full_spatial_flat = probs_full_spatial.numpy().flatten()

# Compute performance metrics
def compute_metrics(y_true, y_pred, model_name):
    """Compute various performance metrics."""
    auc = metrics.roc_auc_score(y_true, y_pred)
    brier = metrics.brier_score_loss(y_true, y_pred)
    ll = metrics.log_loss(y_true, y_pred)
    
    # Binary predictions at 0.5 threshold
    y_pred_binary = (y_pred >= 0.5).astype(int)
    accuracy = metrics.accuracy_score(y_true, y_pred_binary)
    precision = metrics.precision_score(y_true, y_pred_binary, zero_division=0)
    recall = metrics.recall_score(y_true, y_pred_binary, zero_division=0)
    f1 = metrics.f1_score(y_true, y_pred_binary, zero_division=0)
    
    return {
        'Model': model_name,
        'AUC': auc,
        'Brier Score': brier,
        'Log Loss': ll,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }

# Compute metrics for both models
metrics_nonspatial = compute_metrics(Y_flat, probs_full_flat, 'Non-Spatial')
metrics_spatial = compute_metrics(Y_flat, probs_full_spatial_flat, 'Spatial (TPS)')

# Create comparison dataframe
comparison_df = pd.DataFrame([metrics_nonspatial, metrics_spatial])
comparison_df.to_csv(output_dir / 'model_comparison_metrics_spatial.csv', index=False)

print("\n" + "="*70)
print("MODEL PERFORMANCE COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("\nDetailed comparison:")
print(f"\nAUC Improvement (Spatial): {comparison_df.loc[1, 'AUC'] - comparison_df.loc[0, 'AUC']:+.6f}")
print(f"Brier Score Improvement (Spatial, lower is better): {comparison_df.loc[0, 'Brier Score'] - comparison_df.loc[1, 'Brier Score']:+.6f}")
print(f"Log Loss Improvement (Spatial, lower is better): {comparison_df.loc[0, 'Log Loss'] - comparison_df.loc[1, 'Log Loss']:+.6f}")

# Compare predictions
print("\n" + "="*70)
print("PREDICTION COMPARISON")
print("="*70)
correlation = np.corrcoef(probs_full_flat, probs_full_spatial_flat)[0, 1]
print(f"Correlation between non-spatial and spatial predictions: {correlation:.4f}")
print(f"Mean absolute difference: {np.mean(np.abs(probs_full_flat - probs_full_spatial_flat)):.4f}")
print(f"Max absolute difference: {np.max(np.abs(probs_full_flat - probs_full_spatial_flat)):.4f}")



MODEL PERFORMANCE COMPARISON
        Model      AUC  Brier Score  Log Loss  Accuracy  Precision   Recall  F1 Score
  Non-Spatial 0.956131     0.048303  0.159772  0.930889   0.815570 0.393560  0.530920
Spatial (TPS) 0.839488     0.072301  1.109142  0.927333   0.648103 0.588104  0.616647

Detailed comparison:

AUC Improvement (Spatial): -0.116643
Brier Score Improvement (Spatial, lower is better): -0.023998
Log Loss Improvement (Spatial, lower is better): -0.949369

PREDICTION COMPARISON
Correlation between non-spatial and spatial predictions: 0.6254
Mean absolute difference: 0.1082
Max absolute difference: 0.9983


Comparing both approaches, the non-spatial model is significantly better. Therefore, we use the non-spatial model for the main analyses and include the spatial model as a sensitivity check in the supplementary materials.